# Project 2: Fertilizer Optimization Intelligence

This notebook establishes the structure for the fertilizer optimization workstream in the Precision Agriculture and Resource Optimization project. The analysis will be developed incrementally in later steps.

# 1. Business Problem

Farmers often struggle to make fertilizer decisions that are both productive and resource efficient. Common challenges include nutrient imbalance, inefficient fertilizer usage, rising fertilizer costs, soil degradation, and inconsistent crop productivity.

Traditional fertilizer decisions are often based on general recommendations rather than field-specific nutrient conditions. This can make it difficult to match fertilizer strategy to the actual needs of the soil, crop, and local environment.

As a result, resources may be wasted, soil quality can decline, and crop productivity can suffer. This project uses soil nutrient and environmental data to support more informed fertilizer and resource management decisions.

# 2. Project Objective

Project 2 focuses on **Fertilizer Optimization Intelligence**.

The objective is to analyze key soil nutrient and environmental variables, including Nitrogen (N), Phosphorus (P), Potassium (K), pH, rainfall, humidity, and temperature, to generate insights related to nutrient balance, soil health, fertilizer readiness, sustainability, and resource optimization.

This project aims to answer:

> How should nutrients, soil conditions, and resources be managed to support the selected crop?

# 3. Dataset Overview

This project uses the Kaggle Crop Recommendation Dataset:
https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset

The dataset contains soil nutrient measurements, environmental conditions, and crop labels. Its features are directly relevant to fertilizer planning and soil management:

- **N**: Nitrogen level, which supports plant growth and helps assess nitrogen fertilizer needs.
- **P**: Phosphorus level, which is important for root development, crop establishment, and nutrient planning.
- **K**: Potassium level, which supports plant resilience, water regulation, and overall crop health.
- **temperature**: Environmental temperature, which influences crop suitability and nutrient uptake conditions.
- **humidity**: Moisture-related environmental condition that can affect plant stress, disease risk, and resource planning.
- **ph**: Soil acidity or alkalinity, which affects nutrient availability and soil health.
- **rainfall**: Water availability signal that supports irrigation planning and resource-readiness analysis.
- **label**: Crop label used to connect soil and environmental conditions to crop-specific decision support.

Although the dataset was originally designed for crop recommendation, it also provides useful signals for nutrient analysis, fertilizer optimization, and soil health intelligence.

# 4. Exploratory Data Analysis for Fertilizer Optimization Intelligence

This section examines the nutrient and environmental characteristics of the Crop Recommendation Dataset. The goal is to understand patterns that may later support fertilizer optimization, soil health intelligence, and resource readiness analysis.

## Dataset Overview

This first step confirms the dataset structure, field types, missing values, and duplicate rows before interpreting nutrient or environmental patterns.

In [ ]:
# Import libraries for EDA
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import kagglehub
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

plt.style.use("default")
pd.set_option("display.max_columns", 50)

In [ ]:
# Load the Crop Recommendation Dataset from a local copy when available; otherwise use KaggleHub.
local_candidates = [
    Path("Crop_recommendation.csv"),
    Path("data/raw/Crop_recommendation.csv"),
]

data_path = next((candidate for candidate in local_candidates if candidate.exists()), None)

if data_path is None:
    kaggle_dir = Path(kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset"))
    csv_files = sorted(kaggle_dir.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV file found in downloaded Kaggle dataset directory: {kaggle_dir}")
    data_path = csv_files[0]

df = pd.read_csv(data_path)
df.columns = [column.strip().lower() for column in df.columns]

print(f"Loaded dataset from: {data_path}")
df.head()

In [ ]:
required_columns = ["n", "p", "k", "temperature", "humidity", "ph", "rainfall", "label"]
missing_required = [column for column in required_columns if column not in df.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

dataset_overview = pd.DataFrame({
    "metric": ["rows", "columns", "duplicate_rows"],
    "value": [df.shape[0], df.shape[1], df.duplicated().sum()],
})

dataset_overview

In [ ]:
data_types = df[required_columns].dtypes.to_frame("data_type")
missing_values = df[required_columns].isna().sum().to_frame("missing_count")
missing_values["missing_percent"] = (missing_values["missing_count"] / len(df) * 100).round(2)

display(data_types)
display(missing_values)

### Dataset Overview Observation

The dataset overview establishes whether the analysis is working from a complete and usable table. Shape, data types, missing values, and duplicate checks matter because fertilizer and resource-readiness insights depend on consistent nutrient and environmental measurements.

## Nutrient Analysis

This section analyzes Nitrogen (N), Phosphorus (P), and Potassium (K), the core nutrient variables for fertilizer optimization.

In [ ]:
nutrient_cols = ["n", "p", "k"]
df[nutrient_cols].describe().T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, column in zip(axes, nutrient_cols):
    ax.hist(df[column], bins=25, color="#4C78A8", edgecolor="white")
    ax.set_title(f"{column.upper()} Distribution")
    ax.set_xlabel(column.upper())
    ax.set_ylabel("Record Count")

plt.tight_layout()
plt.show()

### Nutrient Histogram Observation

The nutrient histograms show how nitrogen, phosphorus, and potassium values vary across the dataset. Wide or uneven distributions indicate that fertilizer needs are unlikely to be uniform across all crop conditions, which supports the need for fertilizer optimization rather than one-size-fits-all recommendations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, column in zip(axes, nutrient_cols):
    ax.boxplot(df[column], vert=True, patch_artist=True, boxprops={"facecolor": "#72B7B2"})
    ax.set_title(f"{column.upper()} Boxplot")
    ax.set_ylabel(column.upper())
    ax.set_xticks([1])
    ax.set_xticklabels([column.upper()])

plt.tight_layout()
plt.show()

### Nutrient Boxplot Observation

The nutrient boxplots help identify spread, central tendency, and potential outliers in N, P, and K. This matters for fertilizer planning because extreme values may indicate crop-specific nutrient intensity, possible nutrient imbalance, or cases where fertilizer adjustment may be needed.

## Environmental Analysis

This section analyzes temperature, humidity, rainfall, and pH. These variables influence nutrient availability, water readiness, crop suitability, and soil management decisions.

In [ ]:
environment_cols = ["temperature", "humidity", "rainfall", "ph"]
df[environment_cols].describe().T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, column in zip(axes, environment_cols):
    ax.hist(df[column], bins=25, color="#F58518", edgecolor="white")
    ax.set_title(f"{column.title()} Distribution")
    ax.set_xlabel(column)
    ax.set_ylabel("Record Count")

plt.tight_layout()
plt.show()

### Environmental Histogram Observation

The environmental histograms show the range of growing conditions represented in the dataset. Temperature, humidity, rainfall, and pH patterns are important because fertilizer readiness depends not only on nutrient levels but also on whether soil and environmental conditions can support nutrient uptake.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, column in zip(axes, environment_cols):
    ax.boxplot(df[column], vert=True, patch_artist=True, boxprops={"facecolor": "#ECA82C"})
    ax.set_title(f"{column.title()} Boxplot")
    ax.set_ylabel(column)
    ax.set_xticks([1])
    ax.set_xticklabels([column])

plt.tight_layout()
plt.show()

### Environmental Boxplot Observation

The environmental boxplots highlight variability and potential outliers in field conditions. These patterns matter for resource readiness because unusual rainfall, pH, humidity, or temperature values can affect irrigation planning, soil amendments, and the reliability of fertilizer decisions.

## Correlation Analysis

Correlation analysis helps identify relationships among nutrient and environmental variables before any feature engineering or modeling is introduced.

In [ ]:
analysis_cols = nutrient_cols + environment_cols
correlation_matrix = df[analysis_cols].corr(numeric_only=True)
correlation_matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)

ax.set_xticks(np.arange(len(analysis_cols)))
ax.set_yticks(np.arange(len(analysis_cols)))
ax.set_xticklabels(analysis_cols, rotation=45, ha="right")
ax.set_yticklabels(analysis_cols)

for i in range(len(analysis_cols)):
    for j in range(len(analysis_cols)):
        ax.text(j, i, f"{correlation_matrix.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

ax.set_title("Correlation Heatmap: Nutrient and Environmental Variables")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
corr_pairs = (
    correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ["feature_1", "feature_2", "correlation"]
corr_pairs["absolute_correlation"] = corr_pairs["correlation"].abs()
corr_pairs.sort_values("absolute_correlation", ascending=False).reset_index(drop=True)

### Correlation Observation

The heatmap highlights which nutrient and environmental variables move together and which relationships are weak. Stronger relationships may point to shared soil or environmental patterns that could influence fertilizer planning. Weak relationships are also useful because they suggest that some variables may provide distinct signals for later fertilizer readiness and soil health analysis.

## Crop-Level Nutrient Patterns

This section groups the dataset by crop label to compare average N, P, and K levels. The goal is to understand which crops appear more nutrient intensive and which appear less nutrient demanding.

In [ ]:
crop_nutrient_profile = (
    df.groupby("label")[nutrient_cols]
    .mean()
    .round(2)
    .sort_index()
)

crop_nutrient_profile

In [ ]:
ranked_n = crop_nutrient_profile.sort_values("n", ascending=False)
ranked_p = crop_nutrient_profile.sort_values("p", ascending=False)
ranked_k = crop_nutrient_profile.sort_values("k", ascending=False)

print("Top crops by average Nitrogen:")
display(ranked_n.head(10))

print("Top crops by average Phosphorus:")
display(ranked_p.head(10))

print("Top crops by average Potassium:")
display(ranked_k.head(10))

In [ ]:
crop_total_nutrient_rank = (
    crop_nutrient_profile.assign(average_total_npk=crop_nutrient_profile[nutrient_cols].sum(axis=1))
    .sort_values("average_total_npk", ascending=False)
)

crop_total_nutrient_rank.head(10)

In [ ]:
top_crops = crop_total_nutrient_rank.head(10)[nutrient_cols]

ax = top_crops.plot(kind="bar", figsize=(12, 5))
ax.set_title("Top 10 Crops by Average Total NPK")
ax.set_xlabel("Crop")
ax.set_ylabel("Average Nutrient Level")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Crop-Level Nutrient Observation

The ranked tables and bar chart show that nutrient demand appears to vary by crop. Crops with higher average NPK profiles may require closer fertilizer planning and stronger nutrient monitoring, while crops with lower profiles may be less nutrient demanding. This supports a crop-aware fertilizer optimization approach rather than treating all crops as having the same nutrient requirements.

## Initial Project 2 Insights

- Nutrient values vary across the dataset, creating opportunities to analyze fertilizer readiness and nutrient balance in later steps.
- N, P, and K distributions suggest that fertilizer optimization should be crop-aware rather than based on broad general rules.
- Environmental variables such as pH, rainfall, humidity, and temperature provide important soil and resource context for fertilizer planning.
- Crop-level nutrient patterns suggest that some crops may be more nutrient intensive than others, which can inform future soil health and fertilizer readiness indicators.
- Possible future feature engineering opportunities include N:P, N:K, and P:K ratios, total nutrient index, nutrient balance score, and soil health score.

# 5. Feature Engineering Strategy

This section will later define fertilizer-focused features that support nutrient and soil-health intelligence, including:

- N:P ratio
- N:K ratio
- P:K ratio
- total nutrient index
- soil health score
- nutrient balance score

# 6. Fertilizer Optimization Intelligence Framework

This section defines the next intelligence layer for Project 2. It uses the engineered features already established in the broader project, including nutrient ratios, total NPK, water availability, heat-humidity context, nutrient deficiency indicators, and nutrient adjustment features.

This step does not recreate those features or finalize formulas. It establishes the operational framework that later scoring and recommendation logic can build upon.

## 1. Soil Health Intelligence

**Goal:** Develop a Soil Health Score from 0 to 100 that summarizes the overall condition of the soil for crop support and fertilizer planning.

Potential inputs may include nutrient balance, pH suitability, water availability, and environmental conditions. A strong soil health signal should help identify whether the soil is broadly supportive of the selected crop or whether it may require closer nutrient and resource management.

The formula is intentionally not finalized in this step. Later work should define how each input is weighted and validate whether the resulting score is interpretable and useful for decision support.

In [ ]:
# Placeholder: Soil Health Intelligence
# Future work will calculate a Soil Health Score from engineered nutrient,
# pH, water availability, and environmental condition features.

soil_health_inputs = [
    "nutrient_balance_features",
    "ph_suitability_features",
    "water_availability_index",
    "environmental_condition_features",
]

soil_health_inputs

## 2. Fertilizer Readiness Intelligence

**Goal:** Classify fertilizer readiness into operational categories such as **High Readiness**, **Moderate Readiness**, and **Needs Intervention**.

This intelligence layer should use nutrient deficiency and imbalance signals to determine whether the current soil profile appears ready for fertilizer planning or requires intervention before moving forward. For example, major nutrient deficiencies or strong nutrient imbalance may reduce readiness, while balanced nutrient conditions may indicate stronger readiness.

This step defines the classification concept only. Final thresholds and category logic will be developed later.

In [ ]:
# Placeholder: Fertilizer Readiness Intelligence
# Future work will classify readiness using existing deficiency and imbalance indicators.

fertilizer_readiness_categories = [
    "High Readiness",
    "Moderate Readiness",
    "Needs Intervention",
]

fertilizer_readiness_categories

## 3. Sustainability Intelligence

**Goal:** Create a Sustainability Score that reflects nutrient efficiency and environmental suitability.

This layer should consider nutrient overuse risk, soil sustainability, and long-term productivity considerations. In practical terms, sustainability intelligence should help identify whether fertilizer decisions are likely to support crop needs without encouraging unnecessary nutrient excess or soil degradation.

The score is not implemented yet. Future work should define how nutrient efficiency, overuse risk, and environmental suitability contribute to a clear business-facing sustainability indicator.

In [ ]:
# Placeholder: Sustainability Intelligence
# Future work will define a Sustainability Score using nutrient efficiency,
# overuse risk, and environmental suitability signals.

sustainability_inputs = [
    "nutrient_efficiency_signals",
    "nutrient_overuse_risk",
    "soil_sustainability_indicators",
    "environmental_suitability_features",
]

sustainability_inputs

## 4. Resource Readiness Intelligence

**Goal:** Create a Resource Readiness Index based on rainfall, humidity, nutrient status, and environmental suitability.

Potential outputs may include **Resource Ready**, **Resource Constrained**, and **High Risk**. This index should help connect fertilizer planning to broader resource conditions, especially water availability and environmental support.

The intent is to assess whether field conditions appear operationally ready or whether resource constraints may limit the effectiveness of fertilizer decisions.

In [ ]:
# Placeholder: Resource Readiness Intelligence
# Future work will classify resource readiness using rainfall, humidity,
# nutrient status, and environmental suitability.

resource_readiness_categories = [
    "Resource Ready",
    "Resource Constrained",
    "High Risk",
]

resource_readiness_categories

## 5. Fertilizer Recommendation Categories

**Goal:** Generate operational fertilizer recommendation categories such as **Increase Nitrogen**, **Increase Phosphorus**, **Increase Potassium**, **Rebalance Nutrients**, and **Maintain Current Levels**.

These recommendation categories should translate nutrient intelligence into practical decision-support outputs. They can help farmers and planners understand whether a nutrient appears deficient, whether nutrient balance needs attention, or whether current nutrient conditions appear acceptable.

This step does not implement recommendation logic. It defines the category framework for future rule-based or model-supported recommendations.

In [ ]:
# Placeholder: Fertilizer Recommendation Categories
# Future work will map nutrient status and adjustment features into operational categories.

fertilizer_recommendation_categories = [
    "Increase Nitrogen",
    "Increase Phosphorus",
    "Increase Potassium",
    "Rebalance Nutrients",
    "Maintain Current Levels",
]

fertilizer_recommendation_categories

## 6. Business Value

The fertilizer optimization intelligence framework supports practical farming decisions by translating engineered soil and environmental signals into business-facing indicators.

These outputs can support fertilizer planning, soil management, sustainability, resource optimization, and farming readiness. By separating soil health, fertilizer readiness, sustainability, resource readiness, and recommendation categories, the project remains operationally focused while avoiding unsupported claims about exact fertilizer quantities or yield outcomes.

# 7. Soil Health Intelligence

This section will later connect nutrient balance, pH, and environmental conditions to soil-health readiness indicators.

# 8. Modeling Approach

This section will later outline the machine learning approach for supporting fertilizer optimization and soil-health decision support.

# 9. Explainability

This section will later describe how model behavior and recommendation drivers will be explained clearly for agricultural decision-makers.

# 10. Business Recommendations

This section will later translate technical findings into practical recommendations for crop planning, fertilizer management, and resource utilization.

# 11. Future Deployment Considerations

This section will later discuss high-level considerations for using the fertilizer optimization workflow in a realistic decision-support setting.